# Análise Horizontal de Demonstrações Financeiras (Camada Gold)

Este notebook tem como objetivo realizar a análise horizontal do **Balanço Patrimonial** e da **Demonstração de Resultado (DRE)** de instituições financeiras. A análise horizontal avalia a evolução temporal das contas contábeis, permitindo identificar tendências de crescimento ou retração entre dois períodos.

**Etapas do Processo:**
1. Conexão com o Data Warehouse local (`gold_warehouse.duckdb`).
2. Definição dos parâmetros de análise (Instituição e Períodos de comparação).
3. Extração e processamento dos dados estruturados.
4. Geração de relatórios de variação percentual (Ativo, Passivo e Resultado).

In [1]:
# Importações das bibliotecas necessárias para o notebook
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

In [2]:
# Configurações do ambiente para exibição de dados
pd.options.display.float_format = '{:,.2f}'.format

In [3]:
# Configurações do Data Warehouse e parâmetros de consulta
CONFIG: dict[str, str | Path] = {
    # Use pathlib for construct the path to the DuckDB database
    # This resolves the path relative to the current working directory
    'WAREHOUSE_PATH': Path.cwd() / '..' / 'data' / 'gold_warehouse.duckdb',
    'INSTITUICAO': "COOPERATIVA CENTRAL DE CRÉDITO DE MINAS GERAIS LTDA. - SICOOB CENTRAL CREDIMINAS",
    'CURRENT_DATA': '2024-12-01',
    'PREVIOUS_DATA': '2023-12-01',
}

## 1. Funções Auxiliares e Regras de Negócio

Nesta seção, definimos o motor analítico do relatório. As funções abaixo são responsáveis por:

* Estabelecer uma conexão segura (somente leitura) com o banco de dados.
* Formatar os valores numéricos para o padrão monetário brasileiro.
* Calcular a variação percentual entre os períodos, tratando exceções matemáticas (como divisão por zero).
* Orquestrar a geração e a exibição em tela dos blocos contábeis (Ativo, Passivo, DRE).

In [4]:
def get_duckdb_connection(data_warehouse_path: Path) -> duckdb.DuckDBPyConnection | None:
    """Estabelece conexão com o DuckDB com parametros de segurança e performance.

    Args:
        data_warehouse_path (Path): Caminho para o arquivo do DuckDB.

    Returns:
        duckdb.DuckDBPyConnection | None: Conexão estabelecida ou None em caso de erro.
    """

    try:
        connection = duckdb.connect(
            database=str(data_warehouse_path.resolve()),
            read_only=True,  # PoC de análise deve ser read-only para segurança.
            config={'memory_limit': '4GB', 'threads': '4'},
        )
        print(f"Conectado ao Data Warehouse: {data_warehouse_path.name}")
        return connection
    except duckdb.Error as error:
        print(f"Erro ao conectar: {error}")
        return None

In [5]:
def to_brazilian_currency(value):
    """Formata um valor numérico para a notação monetária brasileira."""

    # Transforma o 1000.50 do float para a notação "1.000,50" brasileira.
    return f"{value:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.')

In [6]:
def compute_percentage_change(
    accounts_list: list,
    previous_year_data: pd.Series,
    current_year_data: pd.Series,
    current_year: str,
    previous_year: str,
) -> pd.DataFrame:
    """Processa um bloco de contas, calculando variação percentual entre os anos e formatando o relatório.

    Args:
        accounts_list: Lista de nomes das contas a serem processadas.
        previous_year_data: Série do Pandas contendo os dados do ano anterior.
        current_year_data: Série do Pandas contendo os dados do ano atual.
        current_year: String representando o ano atual (ex: '2024').
        previous_year: String representando o ano anterior (ex: '2023').

    Returns:
        DataFrame do Pandas com as colunas: CONTA, previous_year, current_year, VAR (%).
    """

    relatorio = []
    for account in accounts_list:

        # Pular se a coluna não existir no dataframe,
        # pode acontecer se a instituição não reportar aquela conta específica.
        if account not in current_year_data.index or account not in previous_year_data.index:
            continue

        previous_year_value = float(previous_year_data[account]) if pd.notnull(previous_year_data[account]) else 0.0
        current_year_value = float(current_year_data[account]) if pd.notnull(current_year_data[account]) else 0.0

        # Cálculo de Variação (evita erro de divisão por zero caso no ano anterior não houvesse saldo).
        if previous_year_value == 0:
            percentage_variation = np.nan if current_year_value == 0 else np.inf
        else:
            percentage_variation = ((current_year_value - previous_year_value) / abs(previous_year_value)) * 100

        # Formata o nome da conta (tira underscore e capitaliza).
        nome_conta = account.replace('_', ' ').title()

        relatorio.append(
            {
                'CONTA': nome_conta,
                previous_year: previous_year_value,
                current_year: current_year_value,
                'VAR (%)': percentage_variation,
            }
        )

    return pd.DataFrame(relatorio)

In [7]:
def display_report(
    report_title: str, financial_report_data: pd.DataFrame, current_data: str, previous_data: str
) -> None:
    """Exibe o relatório formatado com um layout elegante.

    Args:
        report_title: Título do relatório a ser exibido.
        financial_report_data: DataFrame contendo as colunas CONTA, previous_data, current_data, VAR (%).
        current_data: String representando o ano atual (ex: '2024').
        previous_data: String representando o ano anterior (ex: '2023').
    """

    print(f"\n{report_title:^95}")
    print("=" * 95)
    print(f"{'CONTA':<45} | {previous_data:>15} | {current_data:>15} | {'VAR (%)':>10}")
    print("-" * 95)

    for _, row in financial_report_data.iterrows():
        account_name = str(row['CONTA'])[:45]
        previous_year_value = to_brazilian_currency(row[previous_data])
        current_year_value = to_brazilian_currency(row[current_data])

        percentage_variation = row['VAR (%)']
        if pd.isna(percentage_variation):
            variation_string = "-"
        elif np.isinf(percentage_variation):
            variation_string = "N/A"
        else:
            variation_string = f"{percentage_variation:+.2f}%"

        print(f"{account_name:<45} | {previous_year_value:>15} | {current_year_value:>15} | {variation_string:>10}")

    print("=" * 95)

In [8]:
def generate_horizontal_analysis(
    financial_data: pd.DataFrame,
    instituicao: str,
    current_period: str,
    previous_period: str,
    report_title: str,
    blocks: dict[str, list[str]],
) -> pd.DataFrame:
    """Gera uma análise horizontal do Balanço Patrimonial entre dois períodos.

    Args:
        balance_sheet_data: DataFrame do Pandas contendo os dados extraídos da view_balanco_patrimonial_relatorios.
        instituicao: String com parte do nome ou código da instituição (ex: 'SICOOB').
        previous_period: String do período base (ex: '2023-12-31' ou '2023').
        current_period: String do período mais recente a ser comparado (ex: '2024-12-31' ou '2024').

    Returns:
        DataFrame do Pandas com as colunas: BLOCO, CONTA, previous_period, current_period, VAR (%).
    """

    df_institution_data = financial_data[
        financial_data['nome_instituicao'].str.contains(instituicao, case=False, na=False)
    ]

    if df_institution_data.empty:
        print(f"❌ Instituição que contenha '{instituicao}' não foi encontrada.")
        return pd.DataFrame()

    df_previous_period = df_institution_data[df_institution_data['data_base'].astype(str).str.contains(previous_period)]
    df_current_period = df_institution_data[df_institution_data['data_base'].astype(str).str.contains(current_period)]

    if df_previous_period.empty or df_current_period.empty:
        print("❌ Dados para uma ou ambas as datas não foram encontrados.")
        print("Datas disponíveis para esta instituição:\n", df_institution_data['data_base'].unique())
        return pd.DataFrame()

    previous_record = df_previous_period.iloc[0]
    current_record = df_current_period.iloc[0]
    institution_name = current_record['nome_instituicao']

    print(f"\n📊 ANÁLISE HORIZONTAL CONTÁBIL: {institution_name.upper()} - {report_title.upper()}")

    dataframes_blocos = []

    for nome_bloco, lista_contas in blocks.items():
        df_bloco = compute_percentage_change(
            lista_contas, previous_record, current_record, current_period, previous_period
        )
        display_report(f">>> {nome_bloco.upper()} <<<", df_bloco, current_period, previous_period)

        df_bloco['BLOCO'] = nome_bloco
        dataframes_blocos.append(df_bloco)

    if not dataframes_blocos:
        return pd.DataFrame()

    df_final = pd.concat(dataframes_blocos, ignore_index=True)

    if not df_final.empty:
        columns = ['BLOCO'] + [col for col in df_final.columns if col != 'BLOCO']
        df_final = df_final[columns]

    return df_final

In [9]:
def generate_horizontal_balance_analysis(
    balance_sheet_data: pd.DataFrame, instituicao: str, current_period: str, previous_period: str
) -> pd.DataFrame:
    """Gera uma análise horizontal do Balanço Patrimonial entre dois períodos, organizando as contas em blocos de Ativo e Passivo/Patrimônio Líquido.

    Args:
        balance_sheet_data (pd.DataFrame): DataFrame contendo os dados do Balanço Patrimonial.
        instituicao (str): Nome da instituição financeira.
        current_period (str): Período atual no formato 'YYYY-MM'.
        previous_period (str): Período anterior no formato 'YYYY-MM'.

    Returns:
        pd.DataFrame: DataFrame com a análise horizontal do Balanço Patrimonial.
    """

    contas_ativo = [
        'disponibilidades',
        'aplicacoes_interfinanceiras_liquidez',
        'tvm_derivativos',
        'operacoes_de_credito',
        'provisao_operacoes_de_credito',
        'operacoes_de_credito_liquidas_provisao',
        'arrendamento_mercantil_a_receber',
        'imobilizado_de_arrendamento',
        'credores_antecipacao_valor_residual',
        'provisao_arrendamento_mercantil',
        'arrendamento_mercantil_liquido_de_provisao',
        'outros_creditos_liquido_de_provisao',
        'outros_ativos_realizaveis',
        'permanente_ajustado',
        'ativo_total_ajustado',
        'ativo_total',
    ]

    contas_passivo = [
        'depositos_vista',
        'depositos_poupanca',
        'depositos_interfinanceiros',
        'depositos_a_prazo',
        'conta_de_pagamento_pre_paga',
        'depositos_outros',
        'deposito_total',
        'captacoes_mercado_aberto',
        'letras_de_credito_imobiliario',
        'letras_de_credito_agronegocio',
        'letras_financeiras',
        'obrigacoes_titulos_e_valores_mobiliarios_exterior',
        'outros_recursos_de_aceites_e_emissao_de_titulos',
        'recursos_aceites_cambiais',
        'obrigacoes_emprestimos_repasses',
        'captacoes_totais',
        'obrigacoes_por_instr_financeiros_derivativos',
        'outras_obrigacoes',
        'passivo_circulante_exigivel_longo_prazo',
        'patrimonio_liquido',
        'passivo_total',
    ]

    blocks = {'Ativo': contas_ativo, 'Passivo e Patrimônio Líquido': contas_passivo}

    generate_horizontal_analysis(
        financial_data=balance_sheet_data,
        instituicao=instituicao,
        current_period=current_period,
        previous_period=previous_period,
        report_title="Balanço Patrimonial",
        blocks=blocks,
    )

In [10]:
def generate_horizontal_dre_analysis(
    dre_data: pd.DataFrame, instituicao: str, current_period: str, previous_period: str
) -> pd.DataFrame:
    """Gera uma análise horizontal da Demonstração de Resultado (DRE) entre dois períodos.

    Args:
        dre_data (pd.DataFrame): DataFrame contendo os dados da DRE.
        instituicao (str): Nome da instituição financeira.
        current_period (str): Período atual no formato 'YYYY-MM'.
        previous_period (str): Período anterior no formato 'YYYY-MM'.

    Returns:
        pd.DataFrame: DataFrame com a análise horizontal da DRE.
    """

    contas_dre = [
        'receitas_intermediacao_financeira',
        'rendas_operacoes_de_credito',
        'rendas_operacoes_de_arrendamento_mercantil',
        'rendas_operacoes_tvm',
        'rendas_operacoes_instrumentos_financeiros_derivativos',
        'resultado_operacoes_cambio',
        'rendas_aplicacoes_compulsorias',
        'despesas_intermediacao_financeira',
        'despesas_captacao',
        'despesas_obrigacoes_emprestimos_repasses',
        'despesas_operacoes_arrendamento_mercantil',
        'despesas_operacoes_cambio',
        'resultado_provisao_creditos_dificil_liquidacao',
        'resultado_intermediacao_financeira',
        'rendas_prestacao_servicos',
        'rendas_tarifas_bancarias',
        'despesas_pessoal',
        'despesas_administrativas',
        'despesas_tributarias',
        'resultado_participacoes',
        'outras_receitas_operacionais',
        'outras_despesas_operacionais',
        'outras_receitas_despesas_operacionais',
        'resultado_operacional',
        'resultado_nao_operacional',
        'resultado_antes_tributacao',
        'imposto_renda_contribuicao_social',
        'participacao_lucros',
        'lucro_liquido',
        'juros_sobre_capital',
    ]

    blocks = {'Resultado (DRE)': contas_dre}

    generate_horizontal_analysis(
        financial_data=dre_data,
        instituicao=instituicao,
        current_period=current_period,
        previous_period=previous_period,
        report_title="Demonstração de Resultado",
        blocks=blocks,
    )

In [11]:
# Inicializa a conexão com o Data Warehouse.
duckdb_connection = get_duckdb_connection(CONFIG['WAREHOUSE_PATH'])

Conectado ao Data Warehouse: gold_warehouse.duckdb


### Contexto dos Dados

Os dados consumidos neste relatório são originários do modelo dimensional (Star Schema) na camada Gold, especificamente:
* **`view_balanco_patrimonial_relatorios`**: Visão consolidada das rubricas patrimoniais.
* **`view_dre_relatorios`**: Visão consolidada das contas de resultado.
* **Filtros aplicados**: `tcb = 'b3c'` (Cooperativas de Crédito) e `tipo_relatorio_bcb = 'Instituições Individuais'`.

## 2. Balanço Patrimonial (Ativo, Passivo e Patrimônio Líquido)

A consulta a seguir extrai os saldos das contas patrimoniais diretamente da `view_balanco_patrimonial_relatorios`. O filtro é aplicado para consolidar apenas "Instituições Individuais" do segmento "b3c".

Após a extração, os dados são divididos em dois grandes blocos (Ativo e Passivo/PL) para o cálculo da variação percentual ano contra ano (YoY) ou período contra período.

In [12]:
balance_sheet_query = """
    SELECT 
        -- IDENTIFICAÇÃO E CONTEXTO
        
        nome_instituicao,
        data_base,
        consolidado_bancario,
        tcb,
        tipo_relatorio_bcb,
        
        -- BLOCO DE ATIVOS (ASSETS)
        
        disponibilidades,
        aplicacoes_interfinanceiras_liquidez,
        tvm_derivativos,
        
        -- Operações de Crédito Analíticas
        operacoes_de_credito,
        provisao_operacoes_de_credito,
        operacoes_de_credito_liquidas_provisao,
        
        -- Arrendamento Mercantil Analítico
        arrendamento_mercantil_a_receber,
        imobilizado_de_arrendamento,
        credores_antecipacao_valor_residual,
        provisao_arrendamento_mercantil,
        arrendamento_mercantil_liquido_de_provisao,
        
        -- Demais Ativos
        outros_creditos_liquido_de_provisao,
        outros_ativos_realizaveis,
        permanente_ajustado,
        
        -- Subtotais e Totais de Ativo
        ativo_total_ajustado,
        ativo_total,
        
        -- BLOCO DE PASSIVOS (LIABILITIES)
        
        -- Depósitos Analíticos
        depositos_vista,
        depositos_poupanca,
        depositos_interfinanceiros,
        depositos_a_prazo,
        conta_de_pagamento_pre_paga,
        depositos_outros,
        depositos AS deposito_total,
        
        -- Captações e Emissões Analíticas
        captacoes_mercado_aberto,
        letras_de_credito_imobiliario,
        letras_de_credito_agronegocio,
        letras_financeiras,
        obrigacoes_titulos_e_valores_mobiliarios_exterior,
        outros_recursos_de_aceites_e_emissao_de_titulos,
        recursos_aceites_cambiais,
        
        -- Repasses e Derivativos
        obrigacoes_emprestimos_repasses,
        captacoes AS captacoes_totais,
        obrigacoes_por_instr_financeiros_derivativos,
        outras_obrigacoes,
        
        -- Subtotais e Totais de Passivo e Patrimônio
        passivo_circulante_exigivel_longo_prazo,
        patrimonio_liquido,
        passivo_total

    FROM
        view_balanco_patrimonial_relatorios

    WHERE
        tipo_relatorio_bcb = 'Instituições Individuais' AND tcb = 'b3c'
      
    ORDER BY
        data_base DESC,
        ativo_total DESC;
"""

In [13]:
if duckdb_connection:
    df_balance_sheet_results = duckdb_connection.sql(balance_sheet_query).df()
    filtered_balance_sheet_2024 = df_balance_sheet_results[df_balance_sheet_results["data_base"] == "2024-12"]
    with pd.option_context("display.max_colwidth", None):
        display(filtered_balance_sheet_2024.sample(5, random_state=42))

,nome_instituicao,data_base,consolidado_bancario,tcb,tipo_relatorio_bcb,disponibilidades,aplicacoes_interfinanceiras_liquidez,tvm_derivativos,operacoes_de_credito,provisao_operacoes_de_credito,...,obrigacoes_titulos_e_valores_mobiliarios_exterior,outros_recursos_de_aceites_e_emissao_de_titulos,recursos_aceites_cambiais,obrigacoes_emprestimos_repasses,captacoes_totais,obrigacoes_por_instr_financeiros_derivativos,outras_obrigacoes,passivo_circulante_exigivel_longo_prazo,patrimonio_liquido,passivo_total
8,"COOPERATIVA CENTRAL DE CRÉDITO, POUPANÇA E INVESTIMENTO DO SUL E SUDESTE - CENTRAL SICREDI SUL/SUDESTE",2024-12-01,Central e Confederação de Cooperativas de Crédito.,b3c,Instituições Individuais,1.00,"335,949.00","4,418,485.00",0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,"7,826,055.00","7,826,055.00",NaN,NaN
13,"COOPERATIVA CENTRAL DE CRÉDITO DE GOIÁS, DISTRITO FEDERAL E TOCANTINS LTDA. - SICOOB NOVA CENTRAL",2024-12-01,Central e Confederação de Cooperativas de Crédito.,b3c,Instituições Individuais,3.00,"1,470,032.00","3,261,685.00","88,878.00",-751.00,...,0.00,0.00,0.00,0.00,0.00,0.00,"4,765,629.00","4,765,629.00",NaN,NaN
9,CENTRAL SICOOB UNI DE COOPERATIVAS DE CRÉDITO,2024-12-01,Central e Confederação de Cooperativas de Crédito.,b3c,Instituições Individuais,18.00,"3,314,222.00","4,148,554.00","169,175.00","-2,438.00",...,0.00,0.00,0.00,0.00,0.00,0.00,"7,456,808.00","7,456,808.00",NaN,NaN
21,CENTRAL DAS COOPERATIVAS DE CRÉDITO DO ESTADO DE SÃO PAULO - SICOOB CENTRAL CECRESP,2024-12-01,Central e Confederação de Cooperativas de Crédito.,b3c,Instituições Individuais,3.00,"435,809.00","1,177,400.00","17,562.00",-57.00,...,0.00,0.00,0.00,0.00,"1,472,419.00",0.00,"136,440.00","1,608,859.00",NaN,NaN
0,COOPERATIVA CENTRAL DE CRÉDITO DE MINAS GERAIS LTDA. - SICOOB CENTRAL CREDIMINAS,2024-12-01,Central e Confederação de Cooperativas de Crédito.,b3c,Instituições Individuais,83.00,"17,565,441.00","7,087,921.00","223,056.00","-2,490.00",...,0.00,0.00,0.00,"434,370.00","462,622.00",0.00,"24,222,631.00","24,685,253.00",NaN,NaN


In [14]:
# Gerar a análise horizontal do Balanço Patrimonial utilizando os dados extraídos do DuckDB.
generate_horizontal_balance_analysis(
    df_balance_sheet_results,
    instituicao=CONFIG['INSTITUICAO'],
    current_period=CONFIG['CURRENT_DATA'],
    previous_period=CONFIG['PREVIOUS_DATA'],
)


📊 ANÁLISE HORIZONTAL CONTÁBIL: COOPERATIVA CENTRAL DE CRÉDITO DE MINAS GERAIS LTDA. - SICOOB CENTRAL CREDIMINAS - BALANÇO PATRIMONIAL

                                         >>> ATIVO <<<                                         
CONTA                                         |      2023-12-01 |      2024-12-01 |    VAR (%)
-----------------------------------------------------------------------------------------------
Disponibilidades                              |           78,00 |           83,00 |     +6.41%
Aplicacoes Interfinanceiras Liquidez          |   11.850.011,00 |   17.565.441,00 |    +48.23%
Tvm Derivativos                               |    6.345.391,00 |    7.087.921,00 |    +11.70%
Operacoes De Credito                          |       64.293,00 |      223.056,00 |   +246.94%
Provisao Operacoes De Credito                 |         -321,00 |       -2.490,00 |   -675.70%
Operacoes De Credito Liquidas Provisao        |       63.972,00 |      220.566,00 |   +244.79%
Arrenda

## 3. Demonstração de Resultado (DRE)

Nesta etapa, o foco muda para o desempenho operacional da instituição. Extraímos as receitas e despesas de intermediação financeira, despesas administrativas e o lucro líquido através da `view_dre_relatorios`. 

A análise horizontal destas contas permite entender a eficiência operacional e a rentabilidade da instituição no período analisado.

In [15]:
dre_query = """
    SELECT 
        nome_instituicao,
        data_base,
        consolidado_bancario,
        tcb,
        tipo_relatorio_bcb,
        
        -- Margens Financeiras
        receitas_intermediacao_financeira,
        rendas_operacoes_de_credito,
        rendas_operacoes_de_arrendamento_mercantil,
        rendas_operacoes_tvm,
        rendas_operacoes_instrumentos_financeiros_derivativos,
        resultado_operacoes_cambio,
        rendas_aplicacoes_compulsorias,
        despesas_intermediacao_financeira,
        despesas_captacao,
        despesas_obrigacoes_emprestimos_repasses,
        despesas_operacoes_arrendamento_mercantil,
        despesas_operacoes_cambio,
        resultado_provisao_creditos_dificil_liquidacao,
        resultado_intermediacao_financeira,
        
        -- Outras Receitas/Despesas Operacionais
        rendas_prestacao_servicos,
        rendas_tarifas_bancarias,
        despesas_pessoal,
        despesas_administrativas,
        despesas_tributarias,
        resultado_participacoes,
        outras_receitas_operacionais,
        outras_despesas_operacionais,
        outras_receitas_despesas_operacionais,
        
        -- Resultados
        resultado_operacional,
        resultado_nao_operacional,
        
        -- Lucro Final
        resultado_antes_tributacao,
        imposto_renda_contribuicao_social,
        participacao_lucros,
        lucro_liquido,
        juros_sobre_capital,
        
        -- Indicadores
        roe,
        roa
    FROM
        view_dre_relatorios
    WHERE
        tipo_relatorio_bcb = 'Instituições Individuais' AND tcb = 'b3c'
    ORDER BY data_base DESC, lucro_liquido DESC;
"""

In [16]:
if duckdb_connection:
    df_dre_query_results = duckdb_connection.sql(dre_query).df()
    filtered_dre_2024 = df_dre_query_results[df_dre_query_results["data_base"] == "2024-12"]
    with pd.option_context("display.max_colwidth", None):
        display(filtered_dre_2024.sample(5, random_state=42))

,nome_instituicao,data_base,consolidado_bancario,tcb,tipo_relatorio_bcb,receitas_intermediacao_financeira,rendas_operacoes_de_credito,rendas_operacoes_de_arrendamento_mercantil,rendas_operacoes_tvm,rendas_operacoes_instrumentos_financeiros_derivativos,...,outras_receitas_despesas_operacionais,resultado_operacional,resultado_nao_operacional,resultado_antes_tributacao,imposto_renda_contribuicao_social,participacao_lucros,lucro_liquido,juros_sobre_capital,roe,roa
8,CENTRAL SICOOB UNI DE COOPERATIVAS DE CRÉDITO,2024-12-01,Central e Confederação de Cooperativas de Crédito.,b3c,Instituições Individuais,"360,827.00",NaN,0.00,NaN,0.00,...,"4,972.00","14,821.00",344.00,"15,166.00",0.00,0.00,"15,166.00",NaN,0.05,0.00
13,CREDISIS - CENTRAL DE COOPERATIVAS DE CRÉDITO LTDA.,2024-12-01,Central e Confederação de Cooperativas de Crédito.,b3c,Instituições Individuais,"105,660.00",NaN,0.00,NaN,0.00,...,"-2,336.00","5,396.00",-10.00,"5,387.00",0.00,0.00,"5,387.00",NaN,0.03,0.00
9,COOPERATIVA CENTRAL DE CRÉDITO DO NORTE DO BRASIL - SICOOB NORTE,2024-12-01,Central e Confederação de Cooperativas de Crédito.,b3c,Instituições Individuais,"212,444.00",NaN,0.00,NaN,0.00,...,"3,858.00","14,732.00",41.00,"14,773.00",-43.00,0.00,"14,730.00",NaN,0.06,0.00
21,"COOPERATIVA CENTRAL DE CRÉDITO, POUPANÇA E INVESTIMENTO DO SUL E SUDESTE - CENTRAL SICREDI SUL/SUDESTE",2024-12-01,Central e Confederação de Cooperativas de Crédito.,b3c,Instituições Individuais,"470,027.00",NaN,0.00,NaN,0.00,...,"-53,317.00",-1.00,1.00,0.00,0.00,0.00,0.00,NaN,0.00,0.00
0,COOPERATIVA CENTRAL DE CRÉDITO DE MINAS GERAIS LTDA. - SICOOB CENTRAL CREDIMINAS,2024-12-01,Central e Confederação de Cooperativas de Crédito.,b3c,Instituições Individuais,"1,294,363.00",NaN,0.00,NaN,0.00,...,"42,745.00","68,749.00",-613.00,"68,136.00",-197.00,"-1,484.00","66,455.00",NaN,0.05,0.00


In [17]:
# Gerar a análise horizontal da DRE utilizando os dados extraídos do DuckDB.
generate_horizontal_dre_analysis(
    df_dre_query_results,
    instituicao=CONFIG['INSTITUICAO'],
    current_period=CONFIG['CURRENT_DATA'],
    previous_period=CONFIG['PREVIOUS_DATA'],
)


📊 ANÁLISE HORIZONTAL CONTÁBIL: COOPERATIVA CENTRAL DE CRÉDITO DE MINAS GERAIS LTDA. - SICOOB CENTRAL CREDIMINAS - DEMONSTRAÇÃO DE RESULTADO

                                    >>> RESULTADO (DRE) <<<                                    
CONTA                                         |      2023-12-01 |      2024-12-01 |    VAR (%)
-----------------------------------------------------------------------------------------------
Receitas Intermediacao Financeira             |    1.098.770,00 |    1.294.363,00 |    +17.80%
Rendas Operacoes De Credito                   |            0,00 |            0,00 |          -
Rendas Operacoes De Arrendamento Mercantil    |            0,00 |            0,00 |          -
Rendas Operacoes Tvm                          |            0,00 |            0,00 |          -
Rendas Operacoes Instrumentos Financeiros Der |            0,00 |            0,00 |          -
Resultado Operacoes Cambio                    |            0,00 |            0,00 |          -
R

In [18]:
# Encerra a conexão com o Data Warehouse para liberar recursos.
duckdb_connection.close()